In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


# First, load data
base_path: str = os.path.dirname(os.path.abspath(__file__))
data_folder_path: str = os.path.join(base_path, "data_test", "raw", "prevalidation_data_2503")
print(data_folder_path)

In [ ]:
import os

print("Current working directory:", os.getcwd())


In [ ]:
import os
os.path.abspath("")

In [ ]:
%pwd


In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
from loguru import logger
from typing import List, Dict, Any, Optional, Union, Tuple

logger.remove()
logger.add(sys.stdout, level="DEBUG")

data_folder_path: str = "/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/prevalidation_data_2503"

def get_file_in_folder(
    folder_path: str = "",
    file_extension: list = None,
    include: list = None,
    exclude: list = None
    ) -> list:
    '''
    Get a list of all files in the directory
    '''
    files = [p for p in Path(data_folder_path).iterdir() if p.is_file()]

    if files and file_extension:
        files = [file for file in files if any(i in str(file.name) for i in file_extension)]

    if files and include:
        files = [file for file in files if any(i in file.name for i in include)]

    if files and exclude:
        files = [file for file in files if not any(i in file.name for i in exclude)]

    if not files:
        return []

    files = [str(file) for file in files]
    return files

In [ ]:
df_file_path: pd.DataFrame = pd.DataFrame([data_folder_path], columns=["folder_path"])
df_file_path

In [ ]:
df_file_path["file_path"] = df_file_path["folder_path"].map(lambda x: get_file_in_folder(folder_path=data_folder_path, file_extension=[".xlsx", ".xls"], exclude=["~$"]))
df_file_path = df_file_path.explode("file_path", ignore_index=True)
df_file_path = df_file_path.sort_values(by=["file_path"], ascending=True).reset_index(drop=True)
df_file_path

In [ ]:
df_file_path["file_name"] = df_file_path["file_path"].astype(str).str.replace("\\", "/")
df_file_path["file_name"] = df_file_path["file_path"].astype(str).str.replace(f"{data_folder_path}/", "")

df_file_path

In [ ]:
# get sheet_names in sheet
import pandas as pd
def get_sheet_names(file_path: str = "", include: list = None, exclude: list = None) -> list:
    sheet_names = pd.ExcelFile(file_path).sheet_names

    if not sheet_names:
        return []

    if include:
        sheet_names = [sheet_name for sheet_name in sheet_names if sheet_name in include]

    if exclude:
        sheet_names = [sheet_name for sheet_name in sheet_names if sheet_name not in exclude]

    return sheet_names

In [ ]:
df_file_path["sheet_name"] = df_file_path["file_path"].map(lambda x: get_sheet_names(file_path=x, exclude=["Config Data"]))
df_file_path = df_file_path.explode("sheet_name", ignore_index=True)

df_file_path = df_file_path.drop(columns=["folder_path"])
df_file_path = df_file_path.sort_values(by=["file_path"])

df_file_path

In [ ]:
import datetime
today_date: str = datetime.datetime.now().strftime("%Y-%m-%d")
parquet_raw_data_folder_path: str = f"/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/parquet/{today_date}"

df_file_path["parquet_file_path"] = df_file_path.apply(lambda x: f"{parquet_raw_data_folder_path}/{x['file_name'].split('.')[0]}/{x['sheet_name']}.parquet", axis=1)
df_file_path.loc[0,:]

In [ ]:
# create file/folder if not exist
import os
import pathlib

@logger.catch
def create_file_or_folder(file_or_folder_path: str = "", is_file: bool = True) -> bool:
    if not file_or_folder_path:
        logger.error("Error: file_or_folder_path is empty.")
        return False

    if os.path.exists(file_or_folder_path):
        logger.info(f"Folder/file already exists: {file_or_folder_path}.")
        return True
    Path(file_or_folder_path).parent.mkdir(parents=True, exist_ok=True)
    if is_file:
        Path(file_or_folder_path).touch()
    logger.success(f"Folder created: {file_or_folder_path}.")
    return True

In [ ]:
# transfer excel to parquet
import polars as pl

@logger.catch
def excel_to_parquet(file_path: str = "", sheet_name: str = "", parquet_file_path: str = ""):
    try:
        pl.read_excel(file_path, sheet_name=sheet_name).write_parquet(parquet_file_path)
        logger.success(f"Excel to Parquet successfully: {file_path} -> {parquet_file_path}.")
        return True
    except Exception as e:
        logger.error(f"Error: {e}")
        return False

df_file_path["is_complete_transfer"] = False

for index, row in df_file_path.iterrows():
    try:
        create_path = create_file_or_folder(file_or_folder_path=row["parquet_file_path"])
        logger.info(create_path)
        if not create_path:
            continue

        if df_file_path.loc[index, "is_complete_transfer"]:
            logger.info(f"File already transfered: {row['file_path']} -> {row['parquet_file_path']}")
            continue

        excel_to_parquet_result: bool = excel_to_parquet(file_path=row["file_path"], sheet_name=row["sheet_name"], parquet_file_path=row["parquet_file_path"])
        logger.info(f"{excel_to_parquet_result}")
        if not excel_to_parquet_result:
            continue

        df_file_path.loc[index, "is_complete_transfer"] = True
    except Exception as e:
        logger.error(f"Error: {e}")

df_file_path

In [ ]:
df_file_path.where(df_file_path["is_complete_transfer"] == False).dropna()

In [ ]:
df_file_path.loc[0, "parquet_file_path"]

In [ ]:
pl.read_parquet(df_file_path.loc[0, "parquet_file_path"])

In [ ]:
df_file_path["file_path"].sort_values(ascending=True).tolist()